# Tiền xử lý dữ liệu cho các mô hình học máy TF-IDF (NLBSE'23 Code Comment Classification)

## 1. Mục tiêu và danh sách các mô hình huấn luyện
Notebook này chuẩn bị dữ liệu đầu vào cho 4 mô hình học máy:
1. **One-vs-Rest Logistic Regression**:	1 classifier nhị phân/nhãn, độc lập
2. **Classifier Chain (Logistic Regression)**:	Chuỗi classifier, nhãn sau dùng dự đoán nhãn trước làm đặc trưng → mô hình hoá tương quan giữa các nhãn
3. **One-vs-Rest Linear SVM**:	1 classifier margin-max/nhãn
4. **One-vs-Rest SGD (log-loss)**: Xấp xỉ logistic regression, huấn luyện online/nhanh hơn

## 2. Quy trình tiền xử lý dữ liệu (Data Preprocessing Pipeline)
- **Bước 1**: Đọc dữ liệu `{language}_preprocessed.csv` (đặt `LANGUAGE=python|java|pharo`) và kiểm tra phân bố nhãn.
- **Bước 2**: Tiền xử lý và chuẩn hóa chuỗi comment (`clean_text`).
- **Bước 3**: Phân chia tập Train/Test 80/20 bằng `MultilabelStratifiedKFold` (cùng cách chia cho Python, Java, Pharo).
- **Bước 4**: Tạo 5-fold Cross-Validation với `MultilabelStratifiedKFold` (bảo toàn tỷ lệ nhãn đa nhãn).
- **Bước 5**: Trích xuất đặc trưng TF-IDF kết hợp Word n-gram (1-2) và Character n-gram (3-5).
- **Bước 6**: Xây dựng ma trận nhãn đa nhãn $Y$ (Binary Relevance format).
- **Bước 7**: Lưu trữ toàn bộ artifacts sẵn sàng cho huấn luyện vào thư mục `data/tfidf/`.
- **Bước 8**: Kiểm tra xác thực tính toàn vẹn của artifacts.

In [ ]:
import os
import re
import json
import pickle
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

# Cấu hình đường dẫn
DATA_DIR = os.path.abspath("../data")
# ── Ngôn ngữ (python | java | pharo) — đặt qua biến môi trường LANGUAGE ────────
LANGUAGE = os.environ.get('LANGUAGE', 'python').lower()
assert LANGUAGE in {'python', 'java', 'pharo'}, f'LANGUAGE không hợp lệ: {LANGUAGE}'
PREPROCESSED_FILE = f'{LANGUAGE}_preprocessed.csv'
NON_LABEL_COLS = {'comment_sentence_id', 'class', 'comment_sentence', 'label_count', 'partition_official'}

OUTPUT_DIR = os.path.join(DATA_DIR, "tfidf") if LANGUAGE == 'python' else os.path.join(DATA_DIR, "tfidf", LANGUAGE)
os.makedirs(OUTPUT_DIR, exist_ok=True)

LABEL_COLS = [c for c in pd.read_csv(os.path.join(DATA_DIR, PREPROCESSED_FILE), nrows=0).columns if c not in NON_LABEL_COLS]
RANDOM_STATE = 42

print(f"Ngôn ngữ: {LANGUAGE}")
print(f"Thư mục dữ liệu: {DATA_DIR}")
print(f"Thư mục lưu artifacts TF-IDF: {OUTPUT_DIR}")
print(f"Danh sách nhãn: {LABEL_COLS}")

## Bước 1: Khám phá và nạp dữ liệu
- Nạp `python_preprocessed.csv` (dạng wide format, 2555 dòng).
- Nạp `python.csv` để lấy chính xác phân vùng partition gốc (train/test).

In [ ]:
df_prep = pd.read_csv(os.path.join(DATA_DIR, PREPROCESSED_FILE))
df_raw = pd.read_csv(os.path.join(DATA_DIR, f"{LANGUAGE}.csv"))

print(f"Kích thước {PREPROCESSED_FILE}: {df_prep.shape}")
print(f"Kích thước {LANGUAGE}.csv (gốc): {df_raw.shape}")
print("\n5 dòng đầu của dữ liệu:")
df_prep.head()

## Bước 2: Chuẩn hóa và làm sạch văn bản comment
- Loại bỏ khoảng trắng thừa, chuẩn hóa ký tự xuống dòng (`\n`, `\t`).
- Giữ nguyên các ký tự code quan trọng vì TF-IDF char n-gram sẽ học từ các ký hiệu này.

In [3]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # Chuẩn hóa xuống dòng và khoảng trắng liên tiếp
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_prep["clean_comment"] = df_prep["comment_sentence"].apply(clean_text)

print("Ví dụ trước và sau chuẩn hóa:")
for orig, cleaned in zip(df_prep["comment_sentence"].head(3), df_prep["clean_comment"].head(3)):
    print(f"- Gốc: {orig}")
    print(f"  Sạch: {cleaned}\n")

Ví dụ trước và sau chuẩn hóa:
- Gốc: abstract cbv mixin that gives access mixins the same customizable
  Sạch: abstract cbv mixin that gives access mixins the same customizable

- Gốc: functionality.
  Sạch: functionality.

- Gốc: more than one migration matches a name prefix.
  Sạch: more than one migration matches a name prefix.



## Bước 3: Phân chia Train/Test (80/20) bằng Multilabel Stratification
Theo phân tích EDA, dữ liệu gốc có 1,231 ID nhận partition khác nhau giữa các category trong dạng long-format (do thiết kế 5 binary tasks riêng biệt). Đối với các mô hình multi-label joint (như Classifier Chains hay One-vs-Rest đa nhãn), việc chia phân tầng trực tiếp trên wide-format () đảm bảo cả 5 nhãn đều có tỷ lệ xuất hiện cân bằng tuyệt đối giữa Train (80%) và Test (20%).

In [4]:
mskf_split = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_dummy = np.zeros(len(df_prep))
Y_all = df_prep[LABEL_COLS].values

# Chọn fold 0 làm Test (20%), 4 folds còn lại làm Train (80%)
train_idx, test_idx = next(mskf_split.split(X_dummy, Y_all))

train_df = df_prep.iloc[train_idx].reset_index(drop=True)
test_df = df_prep.iloc[test_idx].reset_index(drop=True)

print(f"Tập Train: {len(train_df)} mẫu ({len(train_df)/len(df_prep)*100:.2f}%)")
print(f"Tập Test:  {len(test_df)} mẫu ({len(test_df)/len(df_prep)*100:.2f}%)")

split_stats = pd.DataFrame({
    "Train Count": train_df[LABEL_COLS].sum(),
    "Train Ratio (%)": (train_df[LABEL_COLS].mean() * 100).round(2),
    "Test Count": test_df[LABEL_COLS].sum(),
    "Test Ratio (%)": (test_df[LABEL_COLS].mean() * 100).round(2)
})
print("\nPhân bố nhãn giữa tập Train và Test:")
print(split_stats)

Tập Train: 2052 mẫu (80.31%)
Tập Test:  503 mẫu (19.69%)

Phân bố nhãn giữa tập Train và Test:
                  Train Count  Train Ratio (%)  Test Count  Test Ratio (%)
DevelopmentNotes          250            12.18          62           12.33
Expand                    403            19.64         101           20.08
Parameters                635            30.95         159           31.61
Summary                   363            17.69          91           18.09
Usage                     640            31.19         160           31.81


## Bước 4: Tạo 5-Fold Multilabel Stratified Cross-Validation
- Áp dụng `MultilabelStratifiedKFold` từ thư viện `iterstrat` để bảo đảm tỷ lệ xuất hiện của tất cả 5 nhãn cân bằng giữa các fold.

In [5]:
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = []

X_dummy = np.zeros(len(train_df))
Y_train_labels = train_df[LABEL_COLS].values

for fold_idx, (trn_idx, val_idx) in enumerate(mskf.split(X_dummy, Y_train_labels)):
    folds.append({
        'fold': fold_idx,
        'train_indices': trn_idx.tolist(),
        'val_indices': val_idx.tolist()
    })
    print(f"Fold {fold_idx}: Train = {len(trn_idx)} mẫu, Val = {len(val_idx)} mẫu")

folds_path = os.path.join(OUTPUT_DIR, "folds.json")
with open(folds_path, "w", encoding="utf-8") as f:
    json.dump(folds, f, indent=2)
print(f"Đã lưu cấu trúc fold vào: {folds_path}")

Fold 0: Train = 1650 mẫu, Val = 402 mẫu
Fold 1: Train = 1636 mẫu, Val = 416 mẫu
Fold 2: Train = 1649 mẫu, Val = 403 mẫu
Fold 3: Train = 1639 mẫu, Val = 413 mẫu
Fold 4: Train = 1634 mẫu, Val = 418 mẫu
Đã lưu cấu trúc fold vào: /home/linhntn16/projects/introDS/data/tfidf/folds.json


## Bước 5: Trích xuất đặc trưng TF-IDF (Word n-gram + Char n-gram)
- **Word-level TF-IDF (1-2 gram)**: Nắm bắt các từ vựng ngữ nghĩa (`summary`, `parameters`, `returns`, `note`).
- **Character-level TF-IDF (3-5 gram, char_wb)**: Nắm bắt các định danh code, tiền tố, hậu tố, camelCase (`camelCase`, `snake_case`, `param_name`).
- Ghép (hstack) 2 ma trận đặc trưng lại để tạo vector biểu diễn toàn diện.

In [6]:
word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
    sublinear_tf=True
)

char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    max_features=30000,
    min_df=3,
    sublinear_tf=True
)

print("Fit và transform Word TF-IDF...")
X_train_word = word_vectorizer.fit_transform(train_df["clean_comment"])
X_test_word = word_vectorizer.transform(test_df["clean_comment"])

print("Fit và transform Char TF-IDF...")
X_train_char = char_vectorizer.fit_transform(train_df["clean_comment"])
X_test_char = char_vectorizer.transform(test_df["clean_comment"])

X_train = sparse.hstack([X_train_word, X_train_char], format="csr")
X_test = sparse.hstack([X_test_word, X_test_char], format="csr")

print(f"\nKích thước X_train: {X_train.shape}")
print(f"Kích thước X_test:  {X_test.shape}")
sparsity = 100 * (1 - X_train.nnz / (X_train.shape[0] * X_train.shape[1]))
print(f"Độ thưa (Sparsity) của X_train: {sparsity:.2f}%")

Fit và transform Word TF-IDF...
Fit và transform Char TF-IDF...

Kích thước X_train: (2052, 11530)
Kích thước X_test:  (503, 11530)
Độ thưa (Sparsity) của X_train: 99.35%


## Bước 6: Xây dựng ma trận nhãn đa nhãn (Multi-label Matrix $Y$)

In [7]:
y_train = train_df[LABEL_COLS].to_numpy(dtype=np.int8)
y_test = test_df[LABEL_COLS].to_numpy(dtype=np.int8)

print(f"y_train shape: {y_train.shape}, dtype: {y_train.dtype}")
print(f"y_test shape:  {y_test.shape}, dtype: {y_test.dtype}")

y_train shape: (2052, 5), dtype: int8
y_test shape:  (503, 5), dtype: int8


## Bước 7: Lưu trữ các Artifacts vào thư mục `data/tfidf/`
- `X_train.npz`, `X_test.npz`: Ma trận đặc trưng sparse.
- `y_train.npy`, `y_test.npy`: Ma trận nhãn nhị phân.
- `vectorizer_word.pkl`, `vectorizer_char.pkl`: Bộ vectorizer đã fit.
- `label_cols.json`: Tên các cột nhãn theo thứ tự.
- `folds.json`: Chỉ số chia 5 fold.
- `train_meta.csv`, `test_meta.csv`: Metadata kèm văn bản sạch.

In [ ]:
# Lưu ma trận đặc trưng
sparse.save_npz(os.path.join(OUTPUT_DIR, "X_train.npz"), X_train)
sparse.save_npz(os.path.join(OUTPUT_DIR, "X_test.npz"), X_test)

# Lưu ma trận nhãn
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(OUTPUT_DIR, "y_test.npy"), y_test)

# Lưu vectorizers
with open(os.path.join(OUTPUT_DIR, "vectorizer_word.pkl"), "wb") as f:
    pickle.dump(word_vectorizer, f)
with open(os.path.join(OUTPUT_DIR, "vectorizer_char.pkl"), "wb") as f:
    pickle.dump(char_vectorizer, f)

# Lưu tên nhãn
with open(os.path.join(OUTPUT_DIR, "label_cols.json"), "w", encoding="utf-8") as f:
    json.dump(LABEL_COLS, f, indent=2)

# Lưu metadata (partition_official chỉ để đối chiếu, không dùng để chia split)
meta_cols = ["comment_sentence_id", "clean_comment"] + LABEL_COLS
if "partition_official" in train_df.columns:
    meta_cols.append("partition_official")
train_df[meta_cols].to_csv(os.path.join(OUTPUT_DIR, "train_meta.csv"), index=False)
test_df[meta_cols].to_csv(os.path.join(OUTPUT_DIR, "test_meta.csv"), index=False)

print(f"Đã lưu toàn bộ artifacts vào: {OUTPUT_DIR}")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f" - {fname:<25}: {size_kb:>8.2f} KB")

## Bước 8: Kiểm tra tải lại artifacts & Sẵn sàng cho 4 mô hình
Mỗi mô hình trong danh sách đều có thể nạp trực tiếp các dữ liệu đã chuẩn bị sẵn này:

In [9]:
X_tr = sparse.load_npz(os.path.join(OUTPUT_DIR, "X_train.npz"))
X_te = sparse.load_npz(os.path.join(OUTPUT_DIR, "X_test.npz"))
y_tr = np.load(os.path.join(OUTPUT_DIR, "y_train.npy"))
y_te = np.load(os.path.join(OUTPUT_DIR, "y_test.npy"))

assert X_tr.shape == X_train.shape
assert X_te.shape == X_test.shape
assert (y_tr == y_train).all()
assert (y_te == y_test).all()

print("Xác thực dữ liệu thành công! Sẵn sàng huấn luyện 4 mô hình:")
print(f"1. One-vs-Rest Logistic Regression:     X_train {X_tr.shape}, y_train {y_tr.shape}")
print(f"2. Classifier Chain (Logistic Reg):     X_train {X_tr.shape}, y_train {y_tr.shape}")
print(f"3. One-vs-Rest Linear SVM:              X_train {X_tr.shape}, y_train {y_tr.shape}")
print(f"4. One-vs-Rest SGD (log-loss):          X_train {X_tr.shape}, y_train {y_tr.shape}")

Xác thực dữ liệu thành công! Sẵn sàng huấn luyện 4 mô hình:
1. One-vs-Rest Logistic Regression:     X_train (2052, 11530), y_train (2052, 5)
2. Classifier Chain (Logistic Reg):     X_train (2052, 11530), y_train (2052, 5)
3. One-vs-Rest Linear SVM:              X_train (2052, 11530), y_train (2052, 5)
4. One-vs-Rest SGD (log-loss):          X_train (2052, 11530), y_train (2052, 5)
